# Fracture MMS hard-curl PINN — Nonconforming rectangular mesh — Options A and B

This notebook reconstructs the bulk Darcy flux for the single-fracture MMS on
\(\Gamma=\{(t,t):0\le t\le1\}\). The mesh-specific block below is ported from
`LCG_Deng_flux_reconstruction_MMS_rect_nonconforming_P0.ipynb`. The reconstruction itself is imported from
`fracture_hardcurl_common.py` and is mesh-independent: it sees points and oriented
dual-face segments, never elements.

## Fixed conventions

- The fracture is oriented from \(A=(0,0)\) to \(B=(1,1)\), with
  \(\tau=(1,1)/\sqrt2\) and one fixed normal
  \(n_\Gamma=(-1,1)/\sqrt2\), pointing to \(\Omega^+=\{y>x\}\).
- Everywhere,
  \([[q\cdot n_\Gamma]]=q^+\cdot n_\Gamma-q^-\cdot n_\Gamma=\lambda\).
- The bulk balance audited on every dual CV is
  \(\oint_{\partial\omega}q\cdot n\,ds-\int_\omega f_m\,dx
  -\int_{\Gamma\cap\omega}\lambda\,ds=0\).
- Both exact-\(\lambda\) and discrete-\(\lambda_h\) right-hand sides are reported.
- `float64` and fixed seeds are used throughout. Every construction gate prints its
  tolerance and raises immediately on failure.
- The `source` CV class is retained in the common audit schema but is empty/N/A for
  this smooth distributed MMS source; \(\int_\omega f_m\) is still evaluated on
  every CV.
- Because the fracture is corner-to-corner, there is no artificial interface
  extension. Option B reports its extension-continuity penalty explicitly as N/A.

The conforming solve uses the established independent coarse P0 multiplier mesh
with \(h_\lambda=2h\); this is the multiplier used by \(q_{p,\lambda}\).


## 1. Mesh-specific MMS and CG–LMDFM block

Ported without modifying the source notebook.

In [ ]:
# ============================================================
# Configuration and imports
# ============================================================
from mpi4py import MPI
from petsc4py import PETSc
import numpy as np
import math
import time
import pathlib
import basix
import basix.ufl
import matplotlib.pyplot as plt
from tqdm import tqdm

from dolfinx import mesh, fem
import ufl

# Rectangular Deng reconstruction supports Q1/Q2 pressure elements.
order = 1
lambda_order = 0
PRESSURE_ORDER = order
MULTIPLIER_ORDER = lambda_order
FEM_ORDER = PRESSURE_ORDER
Gamma_tag = 2  # compatibility alias; the rectangular mesh has no Gamma facets.

# Independent 1D mesh sizes. The fracture pressure mesh uses h_gamma ~= h/2,
# while the P0 multiplier is coarser to avoid the nonconforming inf-sup trouble.
FRACTURE_PRESSURE_LC_FACTOR = 0.5
LAMBDA_COARSEN = 3.0
MULTIPLIER_LC_FACTOR = LAMBDA_COARSEN

# Section 0: convergence analysis. Keep False for a quick diagnostic run.
RUN_CONVERGENCE = False
# Also run the Deng control-volume reconstruction + LCE/CV diagnostics inside the
# convergence study.  Set False for a fast CG-solve-only convergence run
# (matches the lightweight Section 0 of the nonconforming PINN MMS notebook).
RUN_DENG_RECONSTRUCTION = True
CONV_REFS = [2,3,4,5, 6,7]
CONV_TIP_FRAC = 0.05
CONV_PLOT = True

CASES = [
    {"name": "rect_Q1_P0", "order": 1, "lambda_order": 0},
    {"name": "rect_Q2_P0", "order": 2, "lambda_order": 0},
    {"name": "rect_Q2_P1", "order": 2, "lambda_order": 1}
]
RESULTS_DIR = pathlib.Path("deng_convergence_results/rect_nonconforming_p0")

# Main comparison run.
REF_DEMO = 6

# Jump/error summaries exclude the tip layer below.
TIP_FRAC = CONV_TIP_FRAC

# Residual-map color range. Values outside this range are saturated.
RK_LOG_VMIN = -10.0
RK_LOG_VMAX = -1.0
RK_CBAR_TICKS = np.arange(RK_LOG_VMIN, RK_LOG_VMAX + 1, 1, dtype=float)

Lx, Ly = 1.0, 1.0
x_start, x_end = 0.0, 1.0
y_start, y_end = 0.0, 1.0
FRAC_A = np.array([x_start, y_start], dtype=float)
FRAC_B = np.array([x_end, y_end], dtype=float)
ALPHA = 1.0
K_M_VALUE = 1.0
K_F_VALUE = 100.0

# Manual point-trace assembly is serial.  COMM_SELF also keeps the notebook safe
# when opened from a normal single-kernel Jupyter session.
comm = MPI.COMM_SELF
rank = comm.rank

plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})


In [ ]:
def unit(v):
    v = np.asarray(v, dtype=float)
    nrm = np.linalg.norm(v)
    if nrm == 0.0:
        raise ValueError("zero-length vector")
    return v / nrm


tau_np = unit(FRAC_B - FRAC_A)
normal_np = np.array([-tau_np[1], tau_np[0]], dtype=float)
L_gamma = float(np.linalg.norm(FRAC_B - FRAC_A))


def real_points_np(s):
    s = np.asarray(s, dtype=float).reshape(-1)
    return FRAC_A[None, :] + s[:, None] * tau_np[None, :]


def signed_distance_np(points):
    return (np.asarray(points, dtype=float)[:, :2] - FRAC_A) @ normal_np


def s_coord_np(points):
    return (np.asarray(points, dtype=float)[:, :2] - FRAC_A) @ tau_np


def q_exact_xy(x, y):
    return np.sin(np.pi * x) * np.sin(np.pi * y)


def r_exact_xy(x, y):
    return normal_np[0] * (x - FRAC_A[0]) + normal_np[1] * (y - FRAC_A[1])


def p_m_exact_xy(x, y):
    q = q_exact_xy(x, y)
    return q + ALPHA * np.abs(r_exact_xy(x, y)) * q


def p_gamma_exact_xy(x, y):
    return q_exact_xy(x, y)


def lambda_exact_xy(x, y):
    return -2.0 * ALPHA * q_exact_xy(x, y)



def grad_q_exact_xy(x, y):
    return np.column_stack([
        np.pi * np.cos(np.pi * x) * np.sin(np.pi * y),
        np.pi * np.sin(np.pi * x) * np.cos(np.pi * y),
    ])


def exact_grad_p(points):
    pts = np.asarray(points, dtype=float).reshape(-1, 2)
    x = pts[:, 0]
    y = pts[:, 1]
    q = q_exact_xy(x, y)
    grad_q = grad_q_exact_xy(x, y)
    r = r_exact_xy(x, y)
    sign_r = np.where(r >= 0.0, 1.0, -1.0)
    return grad_q + ALPHA * (
        sign_r[:, None] * normal_np[None, :] * q[:, None]
        + np.abs(r)[:, None] * grad_q
    )


def exact_q(points):
    return -K_M_VALUE * exact_grad_p(points)


def exact_dp_gamma_ds(s):
    pts = real_points_np(s)
    grad_q = grad_q_exact_xy(pts[:, 0], pts[:, 1])
    return grad_q @ tau_np


def f_m_exact_xy(x, y):
    q = q_exact_xy(x, y)
    r = r_exact_xy(x, y)
    sign_r = np.where(r >= 0.0, 1.0, -1.0)
    n_dot_grad_q = (
        normal_np[0] * np.pi * np.cos(np.pi * x) * np.sin(np.pi * y)
        + normal_np[1] * np.pi * np.sin(np.pi * x) * np.cos(np.pi * y)
    )
    return K_M_VALUE * (
        2.0 * np.pi**2 * q
        + 2.0 * ALPHA * np.pi**2 * np.abs(r) * q
        - 2.0 * ALPHA * sign_r * n_dot_grad_q
    )


def f_gamma_exact_xy(x, y):
    qg = q_exact_xy(x, y)
    d2_q_ds2 = (
        -np.pi**2 * qg
        + 2.0 * np.pi**2 * tau_np[0] * tau_np[1]
        * np.cos(np.pi * x) * np.cos(np.pi * y)
    )
    return -K_F_VALUE * d2_q_ds2 + lambda_exact_xy(x, y)


def estimate_bulk_h(omega):
    coords = omega.geometry.x

    def min_positive_spacing(values):
        vals = np.unique(np.round(np.sort(values), 14))
        diffs = np.diff(vals)
        diffs = diffs[diffs > 1.0e-12]
        return float(diffs.min()) if diffs.size else np.inf

    return min(min_positive_spacing(coords[:, 0]), min_positive_spacing(coords[:, 1]))


def make_line_mesh_from_s_nodes(s_nodes):
    s_nodes = np.asarray(s_nodes, dtype=float)
    nodes = real_points_np(s_nodes)
    n_cells = len(s_nodes) - 1
    cells = np.column_stack([np.arange(n_cells), np.arange(1, n_cells + 1)]).astype(np.int64)
    coord_el = ufl.Mesh(basix.ufl.element("Lagrange", basix.CellType.interval, 1, shape=(2,)))
    try:
        gamma_mesh = mesh.create_mesh(comm, cells, coord_el, nodes)
    except (TypeError, AttributeError, IndexError):
        gamma_mesh = mesh.create_mesh(comm, cells, nodes, coord_el)
    return gamma_mesh, float(np.max(np.diff(s_nodes))), int(n_cells), s_nodes


def make_fracture_meshes(h):
    h_gamma_target = FRACTURE_PRESSURE_LC_FACTOR * h
    h_lambda_target = MULTIPLIER_LC_FACTOR * h
    n_gamma = max(1, int(np.ceil(L_gamma / h_gamma_target)))
    n_lambda = max(1, int(np.ceil(L_gamma / h_lambda_target)))
    s_gamma = np.linspace(0.0, L_gamma, n_gamma + 1)
    s_lambda = np.linspace(0.0, L_gamma, n_lambda + 1)
    gamma, h_gamma, n_gamma, _ = make_line_mesh_from_s_nodes(s_gamma)
    gamma_l, h_lambda, n_lambda, _ = make_line_mesh_from_s_nodes(s_lambda)
    return gamma, gamma_l, h_gamma, h_lambda, n_gamma, n_lambda


def make_nonconforming_meshes(ref):
    # Use ref directly so REF_DEMO=6 gives h=1/64, matching the opposite-diagonal mesh.
    n_bulk = 2 ** ref
    omega = mesh.create_unit_square(comm, n_bulk, n_bulk, cell_type=mesh.CellType.quadrilateral)
    h = estimate_bulk_h(omega)
    gamma, gamma_l, h_gamma, h_lambda, n_gamma, n_lambda = make_fracture_meshes(h)
    return omega, gamma, gamma_l, h, h_gamma, h_lambda, n_bulk, n_gamma, n_lambda


def gauss_interval(n):
    x, w = np.polynomial.legendre.leggauss(int(n))
    return 0.5 * (x + 1.0), 0.5 * w


_GB, _WB = gauss_interval(8)
_GL, _WL = gauss_interval(10)


def quad_points_weights():
    pts, weights = [], []
    for i, xi in enumerate(_GB):
        for j, eta in enumerate(_GB):
            pts.append([xi, eta])
            weights.append(_WB[i] * _WB[j])
    return np.asarray(pts, dtype=float), np.asarray(weights, dtype=float)


def _space_ndofs(V):
    return int(V.dofmap.index_map.size_global * V.dofmap.index_map_bs)


# Compatibility callables used by the copied Deng reporting cells.
def p_m_exact_fn(x):
    return p_m_exact_xy(x[0], x[1])


def p_f_exact_fn(t):
    return np.sin(np.pi * t) ** 2


def lam_exact_fn(t):
    return -2.0 * ALPHA * np.sin(np.pi * t) ** 2


def f_m_exact_points(points):
    points = np.asarray(points, dtype=float)
    return f_m_exact_xy(points[:, 0], points[:, 1])


def f_callable(x):
    return f_m_exact_xy(x[0], x[1])


def ff_callable(x):
    return f_gamma_exact_xy(x[0], x[1])


def k_callable(x):
    return (x[0] * 0.0 + K_M_VALUE)[np.newaxis, :]


In [ ]:
def lagrange_1d_values_and_derivatives(nodes, x):
    nodes = np.asarray(nodes, dtype=float).reshape(-1)
    x = np.asarray(x, dtype=float).reshape(-1)
    n = len(nodes)
    vals = np.ones((x.size, n), dtype=float)
    ders = np.zeros((x.size, n), dtype=float)
    if n == 1:
        return vals, ders

    for i in range(n):
        for j in range(n):
            if j != i:
                vals[:, i] *= (x - nodes[j]) / (nodes[i] - nodes[j])
        for k in range(n):
            if k == i:
                continue
            term = np.ones_like(x) / (nodes[i] - nodes[k])
            for j in range(n):
                if j != i and j != k:
                    term *= (x - nodes[j]) / (nodes[i] - nodes[j])
            ders[:, i] += term
    return vals, ders


def polygon_signed_area(poly):
    x = poly[:, 0]
    y = poly[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)


def order_polygon_vertices(pts):
    pts = np.asarray(pts, dtype=float)
    ctr = pts.mean(axis=0)
    ang = np.arctan2(pts[:, 1] - ctr[1], pts[:, 0] - ctr[0])
    poly = pts[np.argsort(ang)]
    if polygon_signed_area(poly) < 0:
        poly = poly[::-1]
    return poly


def build_bulk_cell_data(omega, V_m, n_bulk):
    tdim = omega.topology.dim
    omega.topology.create_connectivity(tdim, 0)
    c2v = omega.topology.connectivity(tdim, 0)
    dof_coords = V_m.tabulate_dof_coordinates()[:, :2]
    geom = omega.geometry.x[:, :2]
    h = 1.0 / n_bulk
    cells = []
    lookup = {}
    num_cells = omega.topology.index_map(tdim).size_local
    for ci in range(num_cells):
        verts = np.asarray(c2v.links(ci), dtype=np.int32)
        poly = order_polygon_vertices(geom[verts])
        x0, y0 = poly.min(axis=0)
        x1, y1 = poly.max(axis=0)
        dofs = np.asarray(V_m.dofmap.cell_dofs(ci), dtype=np.int32)
        dof_xy = dof_coords[dofs]
        nodes_per_dir = int(round(len(dofs) ** 0.5))
        assert nodes_per_dir * nodes_per_dir == len(dofs), "tensor-product Qk cell expected"
        assert nodes_per_dir in (2, 3), "equispaced node assumption only valid for Q1/Q2"
        xi_nodes = np.linspace(0.0, 1.0, nodes_per_dir)
        eta_nodes = np.linspace(0.0, 1.0, nodes_per_dir)
        xi_raw = np.clip((dof_xy[:, 0] - x0) / (x1 - x0), 0.0, 1.0)
        eta_raw = np.clip((dof_xy[:, 1] - y0) / (y1 - y0), 0.0, 1.0)
        dof_xi_idx = np.argmin(np.abs(xi_raw[:, None] - xi_nodes[None, :]), axis=1)
        dof_eta_idx = np.argmin(np.abs(eta_raw[:, None] - eta_nodes[None, :]), axis=1)
        cell = {
            "cell_id": int(ci),
            "dofs": dofs,
            "dof_xy": dof_xy,
            "poly": poly,
            "x0": float(x0),
            "x1": float(x1),
            "y0": float(y0),
            "y1": float(y1),
            "hx": float(x1 - x0),
            "hy": float(y1 - y0),
            "xi_nodes": xi_nodes,
            "eta_nodes": eta_nodes,
            "dof_xi_idx": dof_xi_idx,
            "dof_eta_idx": dof_eta_idx,
            "centroid": np.array([0.5 * (x0 + x1), 0.5 * (y0 + y1)], dtype=float),
        }
        cells.append(cell)
        ix = int(round(x0 / h))
        iy = int(round(y0 / h))
        lookup[(ix, iy)] = int(ci)
    return {"cells": cells, "lookup": lookup, "n_bulk": int(n_bulk), "h": float(h)}


def locate_bulk_cell_xy(pt, bulk_data):
    x, y = np.asarray(pt, dtype=float)[:2]
    n = bulk_data["n_bulk"]
    h = bulk_data["h"]
    tol = 1.0e-12
    ix = n - 1 if x >= 1.0 - tol else int(np.floor(max(x, 0.0) / h))
    iy = n - 1 if y >= 1.0 - tol else int(np.floor(max(y, 0.0) / h))
    ix = min(max(ix, 0), n - 1)
    iy = min(max(iy, 0), n - 1)
    return bulk_data["cells"][bulk_data["lookup"][(ix, iy)]]


def quad_lagrange_values_grads_on_cell(cell, points):
    pts = np.asarray(points, dtype=float).reshape(-1, 2)
    xi = (pts[:, 0] - cell["x0"]) / cell["hx"]
    eta = (pts[:, 1] - cell["y0"]) / cell["hy"]
    Lx, dLx = lagrange_1d_values_and_derivatives(cell["xi_nodes"], xi)
    Ly, dLy = lagrange_1d_values_and_derivatives(cell["eta_nodes"], eta)
    nd = len(cell["dofs"])
    vals = np.zeros((pts.shape[0], nd), dtype=float)
    grads = np.zeros((pts.shape[0], nd, 2), dtype=float)
    for a in range(nd):
        ix = cell["dof_xi_idx"][a]
        iy = cell["dof_eta_idx"][a]
        vals[:, a] = Lx[:, ix] * Ly[:, iy]
        grads[:, a, 0] = (dLx[:, ix] / cell["hx"]) * Ly[:, iy]
        grads[:, a, 1] = Lx[:, ix] * (dLy[:, iy] / cell["hy"])
    return vals, grads


def bulk_breakpoints_on_segment(xa, xb, bulk_data, tol=1.0e-12):
    xa = np.asarray(xa, dtype=float)
    xb = np.asarray(xb, dtype=float)
    d = xb - xa
    n = bulk_data["n_bulk"]
    h = bulk_data["h"]
    out = [0.0, 1.0]
    if abs(d[0]) > tol:
        lo, hi = sorted([xa[0], xb[0]])
        k0 = max(1, int(np.floor(lo / h)) + 1)
        k1 = min(n - 1, int(np.ceil(hi / h)))
        for k in range(k0, k1 + 1):
            r = (k * h - xa[0]) / d[0]
            if tol < r < 1.0 - tol:
                out.append(float(r))
    if abs(d[1]) > tol:
        lo, hi = sorted([xa[1], xb[1]])
        k0 = max(1, int(np.floor(lo / h)) + 1)
        k1 = min(n - 1, int(np.ceil(hi / h)))
        for k in range(k0, k1 + 1):
            r = (k * h - xa[1]) / d[1]
            if tol < r < 1.0 - tol:
                out.append(float(r))
    return np.asarray(sorted(set(np.round(out, 14))), dtype=float)


def build_line_space_data(domain, V):
    domain.topology.create_connectivity(domain.topology.dim, 0)
    c2v = domain.topology.connectivity(domain.topology.dim, 0)
    geom_s = s_coord_np(domain.geometry.x[:, :2])
    dof_coords = V.tabulate_dof_coordinates()[:, :2]
    dof_s_all = s_coord_np(dof_coords)
    cells = []
    s_nodes = []
    for ci in range(domain.topology.index_map(domain.topology.dim).size_local):
        verts = np.asarray(c2v.links(ci), dtype=np.int32)
        s0, s1 = np.sort(geom_s[verts])
        dofs = np.asarray(V.dofmap.cell_dofs(ci), dtype=np.int32)
        dof_s = dof_s_all[dofs]
        ell = float(s1 - s0)
        nodes = np.unique(np.round((dof_s - s0) / ell, 14))
        dof_node_idx = np.searchsorted(nodes, np.round((dof_s - s0) / ell, 14))
        cells.append({
            "cell_id": int(ci),
            "dofs": dofs,
            "s0": float(s0),
            "s1": float(s1),
            "ell": ell,
            "nodes": nodes,
            "dof_node_idx": dof_node_idx,
        })
        s_nodes.extend([float(s0), float(s1)])
    s_nodes = np.array(sorted(set(np.round(s_nodes, 14))), dtype=float)
    cells = [cells[i] for i in np.argsort([c["s0"] for c in cells])]
    return {"cells": cells, "s_nodes": s_nodes}


def locate_line_cell_s(s, line_data, tol=1.0e-12):
    cells = line_data["cells"]
    if s <= cells[0]["s0"] + tol:
        return cells[0]
    if s >= cells[-1]["s1"] - tol:
        return cells[-1]
    starts = np.array([c["s0"] for c in cells], dtype=float)
    idx = int(np.searchsorted(starts, s, side="right") - 1)
    return cells[min(max(idx, 0), len(cells) - 1)]


def line_values_derivatives_on_cell(cell, s):
    s = np.asarray(s, dtype=float).reshape(-1)
    xi = (s - cell["s0"]) / cell["ell"]
    L, dL = lagrange_1d_values_and_derivatives(cell["nodes"], xi)
    nd = len(cell["dofs"])
    vals = np.zeros((s.size, nd), dtype=float)
    ders = np.zeros((s.size, nd), dtype=float)
    for a in range(nd):
        ia = cell["dof_node_idx"][a]
        vals[:, a] = L[:, ia]
        ders[:, a] = dL[:, ia] / cell["ell"]
    return vals, ders


def s_breakpoints(s0, s1, s_nodes, tol=1.0e-12):
    inside = s_nodes[(s_nodes > s0 + tol) & (s_nodes < s1 - tol)]
    return np.concatenate(([s0], inside, [s1])).astype(float)


def edge_length(a, b):
    return float(np.linalg.norm(np.asarray(b, dtype=float) - np.asarray(a, dtype=float)))


def outward_edge_normal(a, b, centroid):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    t = b - a
    n1 = np.array([t[1], -t[0]], dtype=float)
    n2 = -n1
    mid = 0.5 * (a + b)
    chosen = n1 if np.dot(n1, mid - centroid) > 0.0 else n2
    nrm = np.linalg.norm(chosen)
    return chosen / nrm if nrm > 0.0 else chosen


def normal_from_C_to_ti(centroid, midpoint, vi):
    centroid = np.asarray(centroid, dtype=float)
    midpoint = np.asarray(midpoint, dtype=float)
    vi = np.asarray(vi, dtype=float)
    t = midpoint - centroid
    n1 = np.array([t[1], -t[0]], dtype=float)
    n2 = -n1
    seg_mid = 0.5 * (centroid + midpoint)
    chosen = n1 if np.dot(n1, vi - seg_mid) < 0.0 else n2
    nrm = np.linalg.norm(chosen)
    return chosen / nrm if nrm > 0.0 else chosen


def point_in_convex_polygon(pt, poly, tol=1.0e-12):
    pt = np.asarray(pt, dtype=float)
    poly = order_polygon_vertices(np.asarray(poly, dtype=float))
    for j in range(len(poly)):
        a = poly[j]
        b = poly[(j + 1) % len(poly)]
        edge = b - a
        cross = edge[0] * (pt[1] - a[1]) - edge[1] * (pt[0] - a[0])
        if cross < -tol:
            return False
    return True


def segment_edge_s_intersection(edge_a, edge_b, tol=1.0e-12):
    edge_a = np.asarray(edge_a, dtype=float)
    edge_b = np.asarray(edge_b, dtype=float)
    v = edge_b - edge_a
    A = np.column_stack((tau_np, -v))
    det = np.linalg.det(A)
    if abs(det) < tol:
        return None
    rhs = edge_a - FRAC_A
    s, u = np.linalg.solve(A, rhs)
    if -tol <= s <= L_gamma + tol and -tol <= u <= 1.0 + tol:
        return float(np.clip(s, 0.0, L_gamma))
    return None


def fracture_intervals_in_polygon(poly, tol=1.0e-11):
    poly = order_polygon_vertices(np.asarray(poly, dtype=float))
    span = np.ptp(poly, axis=0)
    if abs(polygon_signed_area(poly)) <= tol * tol or np.min(span) <= tol:
        return []
    candidates = [0.0, L_gamma]
    for j in range(len(poly)):
        s_int = segment_edge_s_intersection(poly[j], poly[(j + 1) % len(poly)])
        if s_int is not None:
            candidates.append(s_int)
    candidates = np.array(sorted(candidates), dtype=float)
    unique = []
    for s in candidates:
        if not unique or abs(s - unique[-1]) > tol:
            unique.append(float(s))
    intervals = []
    for s0, s1 in zip(unique[:-1], unique[1:]):
        if s1 - s0 <= tol:
            continue
        sm = 0.5 * (s0 + s1)
        if point_in_convex_polygon(real_points_np([sm])[0], poly, tol=1.0e-10):
            intervals.append((float(s0), float(s1)))
    return intervals


def segment_key(a, b, decimals=12):
    ka = tuple(np.round(np.asarray(a, dtype=float), decimals))
    kb = tuple(np.round(np.asarray(b, dtype=float), decimals))
    return tuple(sorted((ka, kb)))


def assemble_bulk_blocks(A, b, bulk_data):
    qp, qw = quad_points_weights()
    for cell in bulk_data["cells"]:
        x_phys = np.column_stack([
            cell["x0"] + qp[:, 0] * cell["hx"],
            cell["y0"] + qp[:, 1] * cell["hy"],
        ])
        vals, grads = quad_lagrange_values_grads_on_cell(cell, x_phys)
        area = cell["hx"] * cell["hy"]
        Ke = K_M_VALUE * area * np.einsum("q,qid,qjd->ij", qw, grads, grads)
        fe = area * np.einsum("q,q,qi->i", qw, f_m_exact_xy(x_phys[:, 0], x_phys[:, 1]), vals)
        A.setValues(cell["dofs"], cell["dofs"], Ke, addv=PETSc.InsertMode.ADD_VALUES)
        b.setValues(cell["dofs"], fe, addv=PETSc.InsertMode.ADD_VALUES)


def assemble_fracture_and_trace_blocks(A, b, gamma, gamma_l, V_f, V_l, bulk_data, offsets):
    off_m, off_f, off_l = offsets
    pressure_data = build_line_space_data(gamma, V_f)
    multiplier_data = build_line_space_data(gamma_l, V_l)

    # Fracture pressure equation: int K_f dp_f/ds dpsi/ds + int lambda psi = int f_f psi.
    for cell in pressure_data["cells"]:
        fdofs = cell["dofs"]
        s_q = cell["s0"] + cell["ell"] * _GL
        x_q = real_points_np(s_q)
        psi, dpsi = line_values_derivatives_on_cell(cell, s_q)
        weight = cell["ell"] * _WL
        Kloc = K_F_VALUE * np.einsum("q,qi,qj->ij", weight, dpsi, dpsi)
        floc = np.einsum("q,q,qi->i", weight, f_gamma_exact_xy(x_q[:, 0], x_q[:, 1]), psi)
        A.setValues(off_f + fdofs, off_f + fdofs, Kloc, addv=PETSc.InsertMode.ADD_VALUES)
        b.setValues(off_f + fdofs, floc, addv=PETSc.InsertMode.ADD_VALUES)

    for lcell in multiplier_data["cells"]:
        ldofs = lcell["dofs"]
        s0, s1 = lcell["s0"], lcell["s1"]

        pbreaks = s_breakpoints(s0, s1, pressure_data["s_nodes"])
        for a, c in zip(pbreaks[:-1], pbreaks[1:]):
            if c - a <= 1.0e-14:
                continue
            pcell = locate_line_cell_s(0.5 * (a + c), pressure_data)
            fdofs = pcell["dofs"]
            s_q = a + (c - a) * _GL
            psi = line_values_derivatives_on_cell(pcell, s_q)[0]
            mu = line_values_derivatives_on_cell(lcell, s_q)[0]
            weight = (c - a) * _WL
            A.setValues(off_f + fdofs, off_l + ldofs, np.einsum("q,qi,qj->ij", weight, psi, mu), addv=PETSc.InsertMode.ADD_VALUES)
            A.setValues(off_l + ldofs, off_f + fdofs, -np.einsum("q,qi,qj->ij", weight, mu, psi), addv=PETSc.InsertMode.ADD_VALUES)

        xa, xb = real_points_np([s0, s1])
        seg = xb - xa
        breaks = bulk_breakpoints_on_segment(xa, xb, bulk_data)
        for r0, r1 in zip(breaks[:-1], breaks[1:]):
            if r1 - r0 <= 1.0e-14:
                continue
            mid = xa + 0.5 * (r0 + r1) * seg
            bulk_cell = locate_bulk_cell_xy(mid, bulk_data)
            mdofs = bulk_cell["dofs"]
            r_q = r0 + (r1 - r0) * _GL
            s_q = s0 + (s1 - s0) * r_q
            x_q = xa[None, :] + r_q[:, None] * seg[None, :]
            phi = quad_lagrange_values_grads_on_cell(bulk_cell, x_q)[0]
            mu = line_values_derivatives_on_cell(lcell, s_q)[0]
            weight = (s1 - s0) * (r1 - r0) * _WL
            A.setValues(mdofs, off_l + ldofs, -np.einsum("q,qi,qj->ij", weight, phi, mu), addv=PETSc.InsertMode.ADD_VALUES)
            A.setValues(off_l + ldofs, mdofs, np.einsum("q,qi,qj->ij", weight, mu, phi), addv=PETSc.InsertMode.ADD_VALUES)


def apply_pressure_bcs(A, b, V_m, V_f, offsets):
    off_m, off_f, _ = offsets
    coords_m = V_m.tabulate_dof_coordinates()[:, :2]
    coords_f = V_f.tabulate_dof_coordinates()[:, :2]
    tol = 1.0e-12

    bnd_m = np.where(
        np.isclose(coords_m[:, 0], 0.0, atol=tol)
        | np.isclose(coords_m[:, 0], 1.0, atol=tol)
        | np.isclose(coords_m[:, 1], 0.0, atol=tol)
        | np.isclose(coords_m[:, 1], 1.0, atol=tol)
    )[0].astype(np.int32)
    tip_a = np.where(np.linalg.norm(coords_f - FRAC_A[None, :], axis=1) < 100 * tol)[0].astype(np.int32)
    tip_b = np.where(np.linalg.norm(coords_f - FRAC_B[None, :], axis=1) < 100 * tol)[0].astype(np.int32)
    if len(tip_a) != 1 or len(tip_b) != 1:
        raise RuntimeError(f"Could not identify fracture-tip dofs: A={tip_a}, B={tip_b}")

    bc_idx = np.concatenate([off_m + bnd_m, off_f + tip_a, off_f + tip_b]).astype(np.int32)
    bc_vals = np.concatenate([
        p_m_exact_xy(coords_m[bnd_m, 0], coords_m[bnd_m, 1]),
        p_gamma_exact_xy(coords_f[tip_a, 0], coords_f[tip_a, 1]),
        p_gamma_exact_xy(coords_f[tip_b, 0], coords_f[tip_b, 1]),
    ]).astype(float)

    x_bc = A.createVecRight()
    x_bc.set(0.0)
    x_bc.setValues(bc_idx, bc_vals)
    x_bc.assemble()
    A.zeroRowsColumns(bc_idx, diag=1.0, x=x_bc, b=b)
    return bc_idx, bc_vals


def solve_petsc_lu(A, b, prefix):
    x = A.createVecRight()
    x.set(0.0)
    last_error = None
    for factor_solver in ("mumps", None):
        try:
            ksp = PETSc.KSP().create(comm)
            ksp.setOptionsPrefix(prefix)
            ksp.setOperators(A)
            ksp.setType(PETSc.KSP.Type.PREONLY)
            pc = ksp.getPC()
            pc.setType(PETSc.PC.Type.LU)
            if factor_solver is not None:
                pc.setFactorSolverType(factor_solver)
            ksp.setFromOptions()
            ksp.solve(b, x)
            reason = ksp.getConvergedReason()
            if reason < 0:
                raise RuntimeError(f"PETSc KSP failed with reason {reason}")
            return x, factor_solver or "petsc-lu"
        except Exception as exc:
            last_error = exc
            if factor_solver is None:
                raise
    raise RuntimeError("LU solve failed") from last_error


def compute_true_l2_errors(sol, tip_frac=CONV_TIP_FRAC):
    p_m = sol["p_m"]
    p_f = sol["p_f"]
    lmbd = sol["lmbd"]
    bulk_data = sol["bulk_data"]
    pressure_data = build_line_space_data(sol["gamma"], sol["V_f"])
    multiplier_data = build_line_space_data(sol["gamma_l"], sol["V_l"])
    qp, qw = quad_points_weights()

    err_pm_sq = 0.0
    for cell in bulk_data["cells"]:
        x_phys = np.column_stack([
            cell["x0"] + qp[:, 0] * cell["hx"],
            cell["y0"] + qp[:, 1] * cell["hy"],
        ])
        vals = quad_lagrange_values_grads_on_cell(cell, x_phys)[0]
        ph = vals @ p_m.x.array[cell["dofs"]]
        ex = p_m_exact_xy(x_phys[:, 0], x_phys[:, 1])
        err_pm_sq += cell["hx"] * cell["hy"] * float(np.dot(qw, (ph - ex) ** 2))

    err_pf_sq = 0.0
    for cell in pressure_data["cells"]:
        s_q = cell["s0"] + cell["ell"] * _GL
        x_q = real_points_np(s_q)
        psi = line_values_derivatives_on_cell(cell, s_q)[0]
        ph = psi @ p_f.x.array[cell["dofs"]]
        ex = p_gamma_exact_xy(x_q[:, 0], x_q[:, 1])
        err_pf_sq += float(np.dot(cell["ell"] * _WL, (ph - ex) ** 2))

    err_lam_sq = 0.0
    err_lam_int_sq = 0.0
    for cell in multiplier_data["cells"]:
        s_q = cell["s0"] + cell["ell"] * _GL
        x_q = real_points_np(s_q)
        mu = line_values_derivatives_on_cell(cell, s_q)[0]
        lh = mu @ lmbd.x.array[cell["dofs"]]
        ex = lambda_exact_xy(x_q[:, 0], x_q[:, 1])
        diff2 = (lh - ex) ** 2
        weight = cell["ell"] * _WL
        err_lam_sq += float(np.dot(weight, diff2))
        t_q = s_q / L_gamma
        mask = (t_q >= tip_frac) & (t_q <= 1.0 - tip_frac)
        err_lam_int_sq += float(np.dot(weight * mask.astype(float), diff2))

    return {
        "err_pm": np.sqrt(max(err_pm_sq, 0.0)),
        "err_pf": np.sqrt(max(err_pf_sq, 0.0)),
        "err_lam": np.sqrt(max(err_lam_sq, 0.0)),
        "err_lam_int": np.sqrt(max(err_lam_int_sq, 0.0)),
    }


def solve_lcg_nonconforming_mms_2d(ref, pressure_order=PRESSURE_ORDER, multiplier_order=MULTIPLIER_ORDER, return_solution=False, verbose=False):
    omega, gamma, gamma_l, h, h_gamma, h_lambda, n_bulk, n_gamma, n_lambda = make_nonconforming_meshes(ref)
    V_m = fem.functionspace(omega, ("Lagrange", pressure_order))
    V_f = fem.functionspace(gamma, ("Lagrange", pressure_order))
    V_l = fem.functionspace(gamma_l, ("DG", 0)) if multiplier_order == 0 else fem.functionspace(gamma_l, ("Lagrange", multiplier_order))

    n_m = _space_ndofs(V_m)
    n_f = _space_ndofs(V_f)
    n_l = _space_ndofs(V_l)
    offsets = (0, n_m, n_m + n_f)
    n_total = n_m + n_f + n_l

    bulk_data = build_bulk_cell_data(omega, V_m, n_bulk)
    A = PETSc.Mat().createAIJ((n_total, n_total), nnz=240, comm=comm)
    A.setOption(PETSc.Mat.Option.NEW_NONZERO_ALLOCATION_ERR, False)
    A.setUp()
    b = PETSc.Vec().createSeq(n_total, comm=comm)
    b.set(0.0)

    assemble_bulk_blocks(A, b, bulk_data)
    assemble_fracture_and_trace_blocks(A, b, gamma, gamma_l, V_f, V_l, bulk_data, offsets)
    A.assemble()
    b.assemble()
    bc_idx, bc_vals = apply_pressure_bcs(A, b, V_m, V_f, offsets)
    A.assemble()
    b.assemble()

    x, factor_solver = solve_petsc_lu(A, b, prefix=f"mms2d_ref{ref}_p{pressure_order}_l{multiplier_order}_")
    arr = x.getArray(readonly=True)

    p_m = fem.Function(V_m, name="p_m")
    p_f = fem.Function(V_f, name="p_gamma")
    lmbd = fem.Function(V_l, name="lambda_h")
    p_m.x.array[:] = arr[offsets[0]:offsets[1]]
    p_f.x.array[:] = arr[offsets[1]:offsets[2]]
    lmbd.x.array[:] = arr[offsets[2]:offsets[2] + n_l]
    p_m.x.scatter_forward()
    p_f.x.scatter_forward()
    lmbd.x.scatter_forward()

    sol = {
        "ref": int(ref),
        "omega": omega,
        "gamma": gamma,
        "gamma_l": gamma_l,
        "V_m": V_m,
        "V_f": V_f,
        "V_l": V_l,
        "p_m": p_m,
        "p_f": p_f,
        "lmbd": lmbd,
        "h": float(h),
        "h_gamma": float(h_gamma),
        "h_lambda": float(h_lambda),
        "n_bulk": int(n_bulk),
        "n_gamma": int(n_gamma),
        "n_lambda": int(n_lambda),
        "bulk_data": bulk_data,
        "bc_idx": bc_idx,
        "bc_vals": bc_vals,
        "ndof_pm": int(n_m),
        "ndof_pf": int(n_f),
        "ndof_lam": int(n_l),
        "ndof_total": int(n_total),
        "pressure_order": int(pressure_order),
        "multiplier_order": int(multiplier_order),
        "factor_solver": factor_solver,
    }
    sol.update(compute_true_l2_errors(sol, tip_frac=CONV_TIP_FRAC))

    if verbose and rank == 0:
        print(f"PETSc LU factor solver: {factor_solver}")
        print(
            f"n_bulk={n_bulk}, n_gamma={n_gamma}, n_lambda={n_lambda}, "
            f"h={h:.3e}, h_gamma={h_gamma:.3e}, h_lambda={h_lambda:.3e}"
        )
        print(f"dofs: matrix={n_m}, fracture={n_f}, lambda={n_l}, total={n_total}")

    if return_solution:
        return sol
    return {k: sol[k] for k in [
        "ref", "h", "h_gamma", "h_lambda", "n_bulk", "n_gamma", "n_lambda",
        "ndof_pm", "ndof_pf", "ndof_lam", "ndof_total", "pressure_order", "multiplier_order",
        "err_pm", "err_pf", "err_lam", "err_lam_int", "factor_solver",
    ]}



def solve_lcg_mms(ref, verbose=False):
    if int(order) not in (1, 2):
        raise NotImplementedError("Rectangular Deng reconstruction supports Q1/Q2 pressure elements; set order = 1 or 2.")
    sol = solve_lcg_nonconforming_mms_2d(
        ref,
        pressure_order=int(order),
        multiplier_order=int(lambda_order),
        return_solution=True,
        verbose=verbose,
    )
    multiplier_data = build_line_space_data(sol["gamma_l"], sol["V_l"])
    s_lambda = multiplier_data["s_nodes"]

    k_m = fem.Constant(sol["omega"], PETSc.ScalarType(K_M_VALUE))
    f_m = fem.Function(sol["V_m"], name="f_m")
    f_m.interpolate(f_callable)

    ctx = dict(sol)
    ctx.update({
        "V": sol["V_m"],
        "p_sol": sol["p_m"],
        "k_m": k_m,
        "f_m": f_m,
        "Gamma_tag": Gamma_tag,
        "gamma_entities": np.array([], dtype=np.int32),
        "gamma_to_omega": None,
        "gamma_vertex_to_omega": None,
        "cell_markers": None,
        "facet_markers": None,
        "all_dofs": sol["bc_idx"][sol["bc_idx"] < sol["ndof_pm"]].astype(np.int32),
        "iterations": 1,
        "dofs": int(sol["ndof_pm"]),
        "dofs_lambda": int(sol["ndof_lam"]),
        "dofs_total": int(sol["ndof_total"]),
        "lambda_space": "DG0" if int(lambda_order) == 0 else f"P{int(lambda_order)}",
        "lambda_s_nodes": s_lambda,
        "lambda_t_nodes": real_points_np(s_lambda)[:, 0],
        "h_lambda_arc": float(sol["h_lambda"]),
        "lambda_balance_sign": 1.0,
        "err_pm": float(sol["err_pm"]),
        "err_pf": float(sol["err_pf"]),
        "err_lam": float(sol["err_lam"]),
        "mesh_file": "dolfinx.create_unit_square(..., cell_type=mesh.CellType.quadrilateral)",
        "mesh_source": "DOLFINx quadrilateral unit-square mesh",
        "multiplier_data": multiplier_data,
    })
    return ctx


In [ ]:
# ============================================================
# Rectangular geometry and field-evaluation helpers
# ============================================================
def extend_line_to_bbox(a, tau, bbox, tol=1.0e-14):
    xmin, xmax, ymin, ymax = bbox
    ts = []
    if abs(tau[0]) > tol:
        for xval in (xmin, xmax):
            t = (xval - a[0]) / tau[0]
            y = a[1] + t * tau[1]
            if ymin - tol <= y <= ymax + tol:
                ts.append(float(t))
    if abs(tau[1]) > tol:
        for yval in (ymin, ymax):
            t = (yval - a[1]) / tau[1]
            x = a[0] + t * tau[0]
            if xmin - tol <= x <= xmax + tol:
                ts.append(float(t))
    if len(ts) < 2:
        raise RuntimeError("Could not extend fracture line to bounding box.")
    return min(ts), max(ts)

def prepare_fracture_geometry(ctx):
    omega = ctx["omega"]
    tdim = omega.topology.dim
    fdim = tdim - 1
    gdim = omega.geometry.dim
    for pair in [(tdim, 0), (tdim, fdim), (fdim, tdim), (fdim, 0), (0, tdim)]:
        try:
            omega.topology.create_connectivity(*pair)
        except Exception:
            pass

    c2v = omega.topology.connectivity(tdim, 0)
    cell_to_facet = omega.topology.connectivity(tdim, fdim)
    facet_to_cell = omega.topology.connectivity(fdim, tdim)
    facet_to_vertex = omega.topology.connectivity(fdim, 0)

    num_cells = omega.topology.index_map(tdim).size_local
    num_facets = omega.topology.index_map(fdim).size_local
    omega_geometry = omega.geometry.x[:, :2]
    local_cell_vertices = [np.asarray(c2v.links(c), dtype=np.int32) for c in range(num_cells)]
    cell_polys = [order_polygon_vertices(omega_geometry[verts]) for verts in local_cell_vertices]
    cell_centroids = np.vstack([poly.mean(axis=0) for poly in cell_polys])
    cell_areas = np.asarray([abs(polygon_signed_area(poly)) for poly in cell_polys])
    h_est = float(ctx.get("h", np.sqrt(np.mean(cell_areas))))

    coords = omega.geometry.x
    xmin, xmax = coords[:, 0].min(), coords[:, 0].max()
    ymin, ymax = coords[:, 1].min(), coords[:, 1].max()
    t_min, t_max = extend_line_to_bbox(FRAC_A, tau_np, np.array([xmin, xmax, ymin, ymax], dtype=float))
    ghost_start = FRAC_A + t_min * tau_np
    ghost_end = FRAC_A + t_max * tau_np

    def ghost_points_np(n_each=160):
        pts = []
        if -t_min > 1.0e-14:
            s_left = np.linspace(t_min, 0.0, n_each, endpoint=False)
            pts.append(FRAC_A[None, :] + s_left[:, None] * tau_np[None, :])
        if t_max - L_gamma > 1.0e-14:
            s_right = np.linspace(L_gamma, t_max, n_each, endpoint=True)
            pts.append(FRAC_A[None, :] + s_right[:, None] * tau_np[None, :])
        return np.vstack(pts) if pts else np.zeros((0, 2))

    return {
        "tdim": tdim,
        "fdim": fdim,
        "gdim": gdim,
        "c2v": c2v,
        "cell_to_facet": cell_to_facet,
        "facet_to_cell": facet_to_cell,
        "facet_to_vertex": facet_to_vertex,
        "num_cells": num_cells,
        "num_facets": num_facets,
        "omega_geometry": omega_geometry,
        "local_cell_vertices": local_cell_vertices,
        "cell_polys": cell_polys,
        "cell_centroids": cell_centroids,
        "cell_areas": cell_areas,
        "h_est": h_est,
        "facet_by_vertices": {},
        "gamma_facet_set": set(),
        "FRAC_A": FRAC_A.copy(),
        "FRAC_B": FRAC_B.copy(),
        "tau_np": tau_np.copy(),
        "normal_np": normal_np.copy(),
        "L_gamma": L_gamma,
        "GHOST_START": ghost_start,
        "GHOST_END": ghost_end,
        "signed_distance_np": signed_distance_np,
        "s_coord_np": s_coord_np,
        "real_points_np": real_points_np,
        "ghost_points_np": ghost_points_np,
    }


In [ ]:
# ============================================================
# Rectangular multiplier/source helpers
# ============================================================
# ============================================================
# Tensor-product rectangular Deng reconstruction for the nonconforming MMS
# ============================================================
def convergence_rate(errors, hs):
    errors = np.asarray(errors, dtype=float)
    hs = np.asarray(hs, dtype=float)
    rates = [np.nan]
    for e0, e1, h0, h1 in zip(errors[:-1], errors[1:], hs[:-1], hs[1:]):
        if e0 > 0.0 and e1 > 0.0 and h0 > 0.0 and h1 > 0.0 and h0 != h1:
            rates.append(float(np.log(e0 / e1) / np.log(h0 / h1)))
        else:
            rates.append(np.nan)
    return rates


def fmt_rate(value):
    return "  -  " if not np.isfinite(value) else f"{value:5.2f}"


def residual_stats(values, ids=None):
    vals = np.asarray(values if ids is None else np.asarray(values)[ids], dtype=float)
    if vals.size == 0:
        return {"n": 0, "mean": np.nan, "median": np.nan, "max": np.nan}
    vals = np.abs(vals)
    return {"n": int(vals.size), "mean": float(np.mean(vals)), "median": float(np.median(vals)), "max": float(np.max(vals))}


def rect_boundary_vertex_mask(rect, tol=1.0e-12):
    xy = rect["vertices_xy"]
    return (
        np.isclose(xy[:, 0], 0.0, atol=tol)
        | np.isclose(xy[:, 0], 1.0, atol=tol)
        | np.isclose(xy[:, 1], 0.0, atol=tol)
        | np.isclose(xy[:, 1], 1.0, atol=tol)
    )


def physical_edge_defs(cell):
    cached = cell.get("physical_edges")
    if cached is not None:
        return cached
    x0, x1 = float(cell["x0"]), float(cell["x1"])
    y0, y1 = float(cell["y0"]), float(cell["y1"])
    edges = [
        ("bottom", np.array([x0, y0], dtype=float), np.array([x1, y0], dtype=float), np.array([0.0, -1.0], dtype=float)),
        ("right",  np.array([x1, y0], dtype=float), np.array([x1, y1], dtype=float), np.array([1.0, 0.0], dtype=float)),
        ("top",    np.array([x0, y1], dtype=float), np.array([x1, y1], dtype=float), np.array([0.0, 1.0], dtype=float)),
        ("left",   np.array([x0, y0], dtype=float), np.array([x0, y1], dtype=float), np.array([-1.0, 0.0], dtype=float)),
    ]
    return [
        {"side": side, "a": a, "b": b, "normal": n, "key": segment_key(a, b)}
        for side, a, b, n in edges
    ]


def physical_edge_map(cell):
    cached = cell.get("physical_edge_map")
    if cached is not None:
        return cached
    return {edge["side"]: edge for edge in physical_edge_defs(cell)}


def prepare_deng_rect_data(sol):
    pressure_order = int(sol["pressure_order"])
    if pressure_order not in (1, 2):
        raise NotImplementedError(
            "The rectangular Deng reconstruction is implemented for tensor-product Q1/Q2 pressure elements."
        )

    vertex_key_to_id = {}
    vertices_xy = []
    cells = []
    pvals = sol["p_m"].x.array

    def key_xy(xy):
        return tuple(np.round(np.asarray(xy, dtype=float), 12))

    def get_vid(xy):
        key = key_xy(xy)
        if key not in vertex_key_to_id:
            vertex_key_to_id[key] = len(vertices_xy)
            vertices_xy.append(np.asarray(xy, dtype=float))
        return vertex_key_to_id[key]

    for ci, base in enumerate(sol["bulk_data"]["cells"]):
        base_dofs = np.asarray(base["dofs"], dtype=np.int32)
        base_xy = np.asarray(base["dof_xy"], dtype=float)
        vids = np.array([get_vid(xy) for xy in base_xy], dtype=np.int32)
        x0, x1 = float(base["x0"]), float(base["x1"])
        y0, y1 = float(base["y0"]), float(base["y1"])

        cell = {
            "cell_id": int(ci),
            "dofs": base_dofs.copy(),
            "dof_xy": base_xy.copy(),
            "xy": base_xy.copy(),
            "poly": np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], dtype=float),
            "vids": vids,
            "x0": x0,
            "x1": x1,
            "y0": y0,
            "y1": y1,
            "hx": float(base["hx"]),
            "hy": float(base["hy"]),
            "xi_nodes": np.asarray(base["xi_nodes"], dtype=float).copy(),
            "eta_nodes": np.asarray(base["eta_nodes"], dtype=float).copy(),
            "dof_xi_idx": np.asarray(base["dof_xi_idx"], dtype=np.int32).copy(),
            "dof_eta_idx": np.asarray(base["dof_eta_idx"], dtype=np.int32).copy(),
            "centroid": np.asarray(base["centroid"], dtype=float).copy(),
            "uvals": pvals[base_dofs].copy(),
            "pressure_order": pressure_order,
        }
        cell["physical_edges"] = physical_edge_defs(cell)
        cell["physical_edge_map"] = {edge["side"]: edge for edge in cell["physical_edges"]}
        cells.append(cell)

    edge_to_cells = {}
    vtc = [[] for _ in range(len(vertices_xy))]
    for ci, cell_i in enumerate(cells):
        for edge in physical_edge_defs(cell_i):
            edge_to_cells.setdefault(edge["key"], []).append((int(ci), edge["side"]))
        for vid in cell_i["vids"]:
            vtc[int(vid)].append(int(ci))

    for cell in cells:
        nd = len(cell["dofs"])
        cell["cv_bounds"] = [cv_physical_bounds(cell, gi) for gi in range(nd)]
        cell["cv_polygons"] = [cv_polygon(cell, gi) for gi in range(nd)]
        cell["cv_internal_segments"] = [cv_internal_segments(cell, gi) for gi in range(nd)]
        cell["cv_physical_edge_segments"] = [cv_physical_edge_segments(cell, gi) for gi in range(nd)]

    vertices_xy = np.asarray(vertices_xy, dtype=float)
    return {
        "cells": cells,
        "vertices_xy": vertices_xy,
        "edge_to_cells": edge_to_cells,
        "vtc": vtc,
        "edge_defs": ["bottom", "right", "top", "left"],
        "num_cells": len(cells),
        "num_vertices": len(vertices_xy),
    }


def _node_control_bounds(nodes, idx):
    nodes = np.asarray(nodes, dtype=float)
    idx = int(idx)
    lo = 0.0 if idx == 0 else 0.5 * (nodes[idx - 1] + nodes[idx])
    hi = 1.0 if idx == len(nodes) - 1 else 0.5 * (nodes[idx] + nodes[idx + 1])
    return float(lo), float(hi)


def cv_reference_bounds(cell, gi):
    ix = int(cell["dof_xi_idx"][gi])
    iy = int(cell["dof_eta_idx"][gi])
    xi_l, xi_r = _node_control_bounds(cell["xi_nodes"], ix)
    eta_b, eta_t = _node_control_bounds(cell["eta_nodes"], iy)
    return xi_l, xi_r, eta_b, eta_t


def cv_physical_bounds(cell, gi):
    cached = cell.get("cv_bounds")
    if cached is not None:
        return cached[int(gi)]
    xi_l, xi_r, eta_b, eta_t = cv_reference_bounds(cell, gi)
    x_l = cell["x0"] + xi_l * cell["hx"]
    x_r = cell["x0"] + xi_r * cell["hx"]
    y_b = cell["y0"] + eta_b * cell["hy"]
    y_t = cell["y0"] + eta_t * cell["hy"]
    return float(x_l), float(x_r), float(y_b), float(y_t)


def cv_polygon(cell, gi):
    cached = cell.get("cv_polygons")
    if cached is not None:
        return cached[int(gi)]
    x_l, x_r, y_b, y_t = cv_physical_bounds(cell, gi)
    return np.array([[x_l, y_b], [x_r, y_b], [x_r, y_t], [x_l, y_t]], dtype=float)


# Compatibility aliases retained for the older plotting/debug cells.
def standard_cv_polygon(cell, gi):
    return cv_polygon(cell, gi)


def standard_cv_bounds(cell, gi):
    return cv_physical_bounds(cell, gi)


def cv_internal_segments(cell, gi, tol=1.0e-12):
    cached = cell.get("cv_internal_segments")
    if cached is not None:
        return cached[int(gi)]
    xi_l, xi_r, eta_b, eta_t = cv_reference_bounds(cell, gi)
    x_l, x_r, y_b, y_t = cv_physical_bounds(cell, gi)
    segments = []
    if xi_l > tol:
        segments.append((np.array([x_l, y_b], dtype=float), np.array([x_l, y_t], dtype=float), np.array([-1.0, 0.0], dtype=float)))
    if xi_r < 1.0 - tol:
        segments.append((np.array([x_r, y_b], dtype=float), np.array([x_r, y_t], dtype=float), np.array([1.0, 0.0], dtype=float)))
    if eta_b > tol:
        segments.append((np.array([x_l, y_b], dtype=float), np.array([x_r, y_b], dtype=float), np.array([0.0, -1.0], dtype=float)))
    if eta_t < 1.0 - tol:
        segments.append((np.array([x_l, y_t], dtype=float), np.array([x_r, y_t], dtype=float), np.array([0.0, 1.0], dtype=float)))
    return segments


def cv_physical_edge_segments(cell, gi, tol=1.0e-12):
    cached = cell.get("cv_physical_edge_segments")
    if cached is not None:
        return cached[int(gi)]
    xi_l, xi_r, eta_b, eta_t = cv_reference_bounds(cell, gi)
    x_l, x_r, y_b, y_t = cv_physical_bounds(cell, gi)
    edge_map = physical_edge_map(cell)
    segments = []
    if eta_b <= tol and edge_length([x_l, y_b], [x_r, y_b]) > tol:
        edge = edge_map["bottom"]
        segments.append((np.array([x_l, cell["y0"]], dtype=float), np.array([x_r, cell["y0"]], dtype=float), edge["normal"], edge["key"]))
    if eta_t >= 1.0 - tol and edge_length([x_l, y_t], [x_r, y_t]) > tol:
        edge = edge_map["top"]
        segments.append((np.array([x_l, cell["y1"]], dtype=float), np.array([x_r, cell["y1"]], dtype=float), edge["normal"], edge["key"]))
    if xi_l <= tol and edge_length([x_l, y_b], [x_l, y_t]) > tol:
        edge = edge_map["left"]
        segments.append((np.array([cell["x0"], y_b], dtype=float), np.array([cell["x0"], y_t], dtype=float), edge["normal"], edge["key"]))
    if xi_r >= 1.0 - tol and edge_length([x_r, y_b], [x_r, y_t]) > tol:
        edge = edge_map["right"]
        segments.append((np.array([cell["x1"], y_b], dtype=float), np.array([cell["x1"], y_t], dtype=float), edge["normal"], edge["key"]))
    return segments


def _segment_quadrature_points(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return a[None, :] + _GL[:, None] * (b - a)[None, :]


def segment_flux_basis(cell, a, b, n_out):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n_out = np.asarray(n_out, dtype=float)
    seg_len = edge_length(a, b)
    if seg_len == 0.0:
        return np.zeros(len(cell["dofs"]), dtype=float)
    _, grads = quad_lagrange_values_grads_on_cell(cell, _segment_quadrature_points(a, b))
    normal_grads = np.einsum("qjd,d->qj", grads, n_out)
    return -K_M_VALUE * seg_len * np.einsum("q,qj->j", _WL, normal_grads)


def segment_flux_coeffs(cell, coeffs, a, b, n_out):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n_out = np.asarray(n_out, dtype=float)
    seg_len = edge_length(a, b)
    if seg_len == 0.0:
        return 0.0
    _, grads = quad_lagrange_values_grads_on_cell(cell, _segment_quadrature_points(a, b))
    grad = np.einsum("qjd,j->qd", grads, coeffs)
    return float(-K_M_VALUE * seg_len * np.einsum("q,qd,d->", _WL, grad, n_out))


def grad_from_coeffs(cell, coeffs, x):
    pts = np.asarray(x, dtype=float)
    single = pts.ndim == 1
    pts = pts.reshape(-1, 2)
    _, grads = quad_lagrange_values_grads_on_cell(cell, pts)
    grad = np.einsum("qjd,j->qd", grads, coeffs)
    return grad[0] if single else grad


def averaged_kgrad_dot_n(rect, ci, edge_key, x, n_out):
    pts = np.asarray(x, dtype=float)
    single = pts.ndim == 1
    pts = pts.reshape(-1, 2)
    n_out = np.asarray(n_out, dtype=float)
    cell = rect["cells"][ci]
    val = K_M_VALUE * grad_from_coeffs(cell, cell["uvals"], pts)
    adj = [c for c, _ in rect["edge_to_cells"].get(edge_key, [])]
    if len(adj) == 2:
        nb = adj[0] if adj[1] == ci else adj[1]
        nb_cell = rect["cells"][nb]
        val_nb = K_M_VALUE * grad_from_coeffs(nb_cell, nb_cell["uvals"], pts)
        val = 0.5 * (val + val_nb)
    dots = val @ n_out
    return float(dots[0]) if single else dots


def multiplier_s_nodes(sol):
    return build_line_space_data(sol["gamma_l"], sol["V_l"])["s_nodes"]


def lambda_h_on_s_rect(sol, multiplier_data, s_query):
    s = np.asarray(s_query, dtype=float).reshape(-1)
    out = np.zeros_like(s)
    lvals = sol["lmbd"].x.array
    for i, sv in enumerate(s):
        cell = locate_line_cell_s(float(sv), multiplier_data)
        mu = line_values_derivatives_on_cell(cell, np.array([sv]))[0]
        out[i] = float(mu[0] @ lvals[cell["dofs"]])
    return out


def split_at_line_nodes(s0, s1, s_nodes, tol=1.0e-12):
    lo, hi = (float(s0), float(s1)) if s0 <= s1 else (float(s1), float(s0))
    inner = s_nodes[(s_nodes > lo + tol) & (s_nodes < hi - tol)]
    knots = np.concatenate(([lo], inner, [hi]))
    return list(zip(knots[:-1], knots[1:]))


def integrate_lambda_interval(sol, multiplier_data, s0, s1):
    total = 0.0
    for a, b in split_at_line_nodes(s0, s1, multiplier_data["s_nodes"]):
        if b - a <= 1.0e-14:
            continue
        s_q = a + (b - a) * _GL
        total += float(np.dot((b - a) * _WL, lambda_h_on_s_rect(sol, multiplier_data, s_q)))
    return total


def integrate_lambda_phi_interval(sol, multiplier_data, cell, s0, s1):
    out = np.zeros(len(cell["dofs"]), dtype=float)
    for a, b in split_at_line_nodes(s0, s1, multiplier_data["s_nodes"]):
        if b - a <= 1.0e-14:
            continue
        s_q = a + (b - a) * _GL
        pts = real_points_np(s_q)
        lam = lambda_h_on_s_rect(sol, multiplier_data, s_q)
        phi = quad_lagrange_values_grads_on_cell(cell, pts)[0]
        out += np.einsum("q,q,qi->i", (b - a) * _WL, lam, phi)
    return out


def integrate_f_over_rect(x_l, x_r, y_b, y_t):
    x_q = x_l + (x_r - x_l) * _GB
    y_q = y_b + (y_t - y_b) * _GB
    X, Y = np.meshgrid(x_q, y_q, indexing="ij")
    F = np.asarray(f_m_exact_xy(X.ravel(), Y.ravel()), dtype=float).reshape(X.shape)
    total = float(np.einsum("i,j,ij->", _WB, _WB, F))
    return float((x_r - x_l) * (y_t - y_b) * total)


def cell_volume_terms(cell):
    qp, qw = quad_points_weights()
    x_phys = np.column_stack([
        cell["x0"] + qp[:, 0] * cell["hx"],
        cell["y0"] + qp[:, 1] * cell["hy"],
    ])
    phi, grads = quad_lagrange_values_grads_on_cell(cell, x_phys)
    area = cell["hx"] * cell["hy"]
    fvals = f_m_exact_xy(x_phys[:, 0], x_phys[:, 1])
    grad_u = np.einsum("qjd,j->qd", grads, cell["uvals"])
    source_phi = area * np.einsum("q,q,qi->i", qw, fvals, phi)
    a_term = area * np.einsum("q,qjd,qd->j", qw, K_M_VALUE * grads, grad_u)
    mass_vec = area * np.einsum("q,qi->i", qw, phi)
    return source_phi, a_term, mass_vec


In [ ]:
# ============================================================
# Deng local post-processing on tensor-product rectangular control volumes
# ============================================================
def solve_deng_singular_system(B, rhs, rcond=1.0e-10):
    U, s, Vt = np.linalg.svd(B, full_matrices=False)
    if s.size == 0 or s[0] == 0.0:
        return np.zeros(B.shape[1], dtype=float), 0, s
    keep = s > rcond * s[0]
    coeffs = Vt[keep, :].T @ ((U[:, keep].T @ rhs) / s[keep])
    return coeffs, int(np.count_nonzero(keep)), s


def deng_postprocess_rect_nonconforming(sol):
    rect = prepare_deng_rect_data(sol)
    multiplier_data = build_line_space_data(sol["gamma_l"], sol["V_l"])
    local_fluxes = {}
    coef_glob = {i: [] for i in range(rect["num_vertices"])}

    for ci, cell in enumerate(rect["cells"]):
        nd = len(cell["dofs"])
        B = np.zeros((nd, nd), dtype=float)
        source_t_matrix = np.zeros(nd, dtype=float)
        source_phi_matrix, a_term, mass_vec = cell_volume_terms(cell)
        source_t_gamma = np.zeros(nd, dtype=float)
        source_phi_gamma = np.zeros(nd, dtype=float)
        e_I = np.zeros(nd, dtype=float)
        e_phi = np.zeros(nd, dtype=float)

        cell_gamma_intervals = fracture_intervals_in_polygon(cell["poly"])
        for gi in range(nd):
            x_l, x_r, y_b, y_t = cv_physical_bounds(cell, gi)
            source_t_matrix[gi] += integrate_f_over_rect(x_l, x_r, y_b, y_t)

            for s0, s1 in fracture_intervals_in_polygon(cv_polygon(cell, gi)):
                source_t_gamma[gi] += integrate_lambda_interval(sol, multiplier_data, s0, s1)

            for a, b, n_out in cv_internal_segments(cell, gi):
                B[gi, :] += segment_flux_basis(cell, a, b, n_out)

        for s0, s1 in cell_gamma_intervals:
            source_phi_gamma += integrate_lambda_phi_interval(sol, multiplier_data, cell, s0, s1)

        for edge in physical_edge_defs(cell):
            a = edge["a"]
            b = edge["b"]
            n_out = edge["normal"]
            edge_key = edge["key"]
            seg_len = edge_length(a, b)
            pts = _segment_quadrature_points(a, b)
            phi_edge = quad_lagrange_values_grads_on_cell(cell, pts)[0]
            avg = averaged_kgrad_dot_n(rect, ci, edge_key, pts, n_out)
            e_phi += seg_len * np.einsum("q,q,qi->i", _WL, avg, phi_edge)

        for gi in range(nd):
            for a, b, n_out, edge_key in cv_physical_edge_segments(cell, gi):
                seg_len = edge_length(a, b)
                pts = _segment_quadrature_points(a, b)
                avg = averaged_kgrad_dot_n(rect, ci, edge_key, pts, n_out)
                e_I[gi] += seg_len * float(np.dot(_WL, avg))

        # B has the constants in its nullspace, so the discrete CV sources must
        # have the same total as the FE load quadrature used in the weak solve.
        source_t_matrix_raw = source_t_matrix.copy()
        source_defect = float(np.sum(source_phi_matrix + source_phi_gamma) - np.sum(source_t_matrix + source_t_gamma))
        area = float(np.sum(mass_vec))
        if area > 0.0:
            source_t_matrix += source_defect * mass_vec / area

        source_t = source_t_matrix + source_t_gamma
        source_phi = source_phi_matrix + source_phi_gamma
        rhs = source_t - source_phi + a_term + e_I - e_phi

        coeffs, local_solve_rank, local_solve_svals = solve_deng_singular_system(B, rhs)

        area = float(np.sum(mass_vec))
        if area > 0.0:
            coeffs += (float(np.dot(mass_vec, cell["uvals"])) - float(np.dot(mass_vec, coeffs))) / area

        local_fluxes[ci] = {
            "coeffs": coeffs,
            "B": B.copy(),
            "source_t": source_t.copy(),
            "source_phi": source_phi.copy(),
            "source_t_matrix": source_t_matrix.copy(),
            "source_t_matrix_raw": source_t_matrix_raw.copy(),
            "source_compat_defect": source_defect,
            "source_phi_matrix": source_phi_matrix.copy(),
            "source_t_gamma": source_t_gamma.copy(),
            "source_phi_gamma": source_phi_gamma.copy(),
            "rhs": rhs.copy(),
            "local_solve_rank": int(local_solve_rank),
            "local_solve_singular_values": local_solve_svals.copy(),
        }
        for li, vid in enumerate(cell["vids"]):
            coef_glob[int(vid)].append(coeffs[li])

    p_rec = np.zeros(rect["num_vertices"], dtype=float)
    for vid, vals in coef_glob.items():
        p_rec[vid] = float(np.mean(vals)) if vals else np.nan
    return rect, multiplier_data, local_fluxes, coef_glob, p_rec



def deng_postprocess_fracture(ctx):
    rect, multiplier_data, local_fluxes, coef_glob, p_rec = deng_postprocess_rect_nonconforming(ctx)
    ctx["rect"] = rect
    ctx["multiplier_data"] = multiplier_data
    ctx["p_rec"] = p_rec
    return local_fluxes, coef_glob, p_rec


In [ ]:
# ============================================================
# Residuals, errors, rates, and plotting helpers
# ============================================================
def compute_lce_per_element_rect(rect, local_fluxes, use_rec=True):
    R = np.zeros(rect["num_cells"], dtype=float)
    for ci, cell in enumerate(rect["cells"]):
        coeffs = local_fluxes[ci]["coeffs"] if use_rec else cell["uvals"]
        flux = 0.0
        for edge in physical_edge_defs(cell):
            flux += segment_flux_coeffs(cell, coeffs, edge["a"], edge["b"], edge["normal"])
        R[ci] = flux - float(np.sum(local_fluxes[ci]["source_t"]))
    return R


def compute_lce_per_cv_rect(rect, local_fluxes, use_rec=True):
    R = np.zeros(rect["num_vertices"], dtype=float)
    lambda_cv = np.zeros(rect["num_vertices"], dtype=float)
    fracture_vertex = np.zeros(rect["num_vertices"], dtype=bool)
    for vid in range(rect["num_vertices"]):
        flux_sum = 0.0
        src_sum = 0.0
        for ci in rect["vtc"][vid]:
            cell = rect["cells"][ci]
            loc = np.where(cell["vids"] == vid)[0]
            if len(loc) == 0:
                continue
            gi = int(loc[0])
            coeffs = local_fluxes[ci]["coeffs"] if use_rec else cell["uvals"]
            for a, b, n_out in cv_internal_segments(cell, gi):
                flux_sum += segment_flux_coeffs(cell, coeffs, a, b, n_out)
            src_sum += local_fluxes[ci]["source_t"][gi]
            lambda_cv[vid] += local_fluxes[ci]["source_t_gamma"][gi]
            if abs(local_fluxes[ci]["source_t_gamma"][gi]) > 1.0e-14:
                fracture_vertex[vid] = True
        R[vid] = flux_sum - src_sum
    return R, lambda_cv, fracture_vertex

def compute_bulk_pressure_flux_errors(sol, rect, local_fluxes, use_rec=False):
    qp, qw = quad_points_weights()
    l2_sq = 0.0
    h1_sq = 0.0
    q_sq = 0.0
    for ci, cell in enumerate(rect["cells"]):
        coeffs = local_fluxes[ci]["coeffs"] if use_rec else cell["uvals"]
        x_phys = np.column_stack([
            cell["x0"] + qp[:, 0] * cell["hx"],
            cell["y0"] + qp[:, 1] * cell["hy"],
        ])
        phi, grads = quad_lagrange_values_grads_on_cell(cell, x_phys)
        p_num = phi @ coeffs
        grad_num = np.einsum("qjd,j->qd", grads, coeffs)
        p_ex = p_m_exact_xy(x_phys[:, 0], x_phys[:, 1])
        grad_ex = exact_grad_p(x_phys)
        q_num = -K_M_VALUE * grad_num
        q_ex = -K_M_VALUE * grad_ex
        area = cell["hx"] * cell["hy"]
        l2_sq += area * float(np.dot(qw, (p_num - p_ex) ** 2))
        h1_sq += area * float(np.dot(qw, np.sum((grad_num - grad_ex) ** 2, axis=1)))
        q_sq += area * float(np.dot(qw, np.sum((q_num - q_ex) ** 2, axis=1)))
    return np.sqrt(max(l2_sq, 0.0)), np.sqrt(max(h1_sq, 0.0)), np.sqrt(max(q_sq, 0.0))


def compute_fracture_pressure_errors(sol):
    pressure_data = build_line_space_data(sol["gamma"], sol["V_f"])
    p_f = sol["p_f"].x.array
    l2_sq = 0.0
    h1_sq = 0.0
    for cell in pressure_data["cells"]:
        s_q = cell["s0"] + cell["ell"] * _GL
        x_q = real_points_np(s_q)
        psi, dpsi = line_values_derivatives_on_cell(cell, s_q)
        ph = psi @ p_f[cell["dofs"]]
        dph = dpsi @ p_f[cell["dofs"]]
        ex = p_gamma_exact_xy(x_q[:, 0], x_q[:, 1])
        dex = exact_dp_gamma_ds(s_q)
        weight = cell["ell"] * _WL
        l2_sq += float(np.dot(weight, (ph - ex) ** 2))
        h1_sq += float(np.dot(weight, (dph - dex) ** 2))
    return np.sqrt(max(l2_sq, 0.0)), np.sqrt(max(h1_sq, 0.0))


def compute_lambda_l2_errors(sol, multiplier_data, tip_frac=CONV_TIP_FRAC):
    full = 0.0
    interior = 0.0
    for cell in multiplier_data["cells"]:
        s_q = cell["s0"] + cell["ell"] * _GL
        x_q = real_points_np(s_q)
        lh = lambda_h_on_s_rect(sol, multiplier_data, s_q)
        ex = lambda_exact_xy(x_q[:, 0], x_q[:, 1])
        diff2 = (lh - ex) ** 2
        weight = cell["ell"] * _WL
        full += float(np.dot(weight, diff2))
        t_q = s_q / L_gamma
        mask = (t_q >= tip_frac) & (t_q <= 1.0 - tip_frac)
        interior += float(np.dot(weight * mask.astype(float), diff2))
    return np.sqrt(max(full, 0.0)), np.sqrt(max(interior, 0.0))


def q_bulk_at_points(sol, rect, local_fluxes, points, use_rec=False):
    pts = np.asarray(points, dtype=float).reshape(-1, 2)
    out = np.zeros((pts.shape[0], 2), dtype=float)
    for i, pt in enumerate(pts):
        cell_base = locate_bulk_cell_xy(pt, sol["bulk_data"])
        ci = int(cell_base["cell_id"])
        cell = rect["cells"][ci]
        coeffs = local_fluxes[ci]["coeffs"] if use_rec else cell["uvals"]
        out[i] = -K_M_VALUE * grad_from_coeffs(cell, coeffs, pt)
    return out


def compute_lambda_jump_l2_rect(sol, rect, local_fluxes, multiplier_data, use_rec=False, tip_frac=CONV_TIP_FRAC, eps=None):
    if eps is None:
        eps = 0.1 * sol["h"]
    s_nodes = multiplier_data["s_nodes"]
    if len(s_nodes) < 2:
        return np.nan
    s_mid = 0.5 * (s_nodes[:-1] + s_nodes[1:])
    ds = np.diff(s_nodes)
    mask = (s_mid >= tip_frac * L_gamma) & (s_mid <= (1.0 - tip_frac) * L_gamma)
    if not np.any(mask):
        return np.nan
    x_mid = real_points_np(s_mid)
    qp = q_bulk_at_points(sol, rect, local_fluxes, x_mid + eps * normal_np[None, :], use_rec=use_rec)
    qm = q_bulk_at_points(sol, rect, local_fluxes, x_mid - eps * normal_np[None, :], use_rec=use_rec)
    jump = np.sum((qp - qm) * normal_np[None, :], axis=1)
    exact = lambda_exact_xy(x_mid[:, 0], x_mid[:, 1])
    return np.sqrt(max(float(np.sum(((jump - exact) ** 2) * ds * mask.astype(float))), 0.0))




# Keep the copied triangular notebook's public function names, but route them to
# the tensor-product rectangular control-volume implementation above.
_compute_fracture_pressure_errors_rect = compute_fracture_pressure_errors


def _ctx_rect(ctx):
    if "rect" not in ctx:
        ctx["rect"] = prepare_deng_rect_data(ctx)
    return ctx["rect"]


def _ctx_multiplier_data(ctx):
    if "multiplier_data" not in ctx:
        ctx["multiplier_data"] = build_line_space_data(ctx["gamma_l"], ctx["V_l"])
    return ctx["multiplier_data"]


def lambda_h_on_s(ctx, gdata, s_query=None):
    if s_query is None:
        s_query = gdata
    return lambda_h_on_s_rect(ctx, _ctx_multiplier_data(ctx), s_query)


def q_cg_numpy(ctx, gdata, points):
    return q_bulk_at_points(ctx, _ctx_rect(ctx), {}, points, use_rec=False)


def q_rec_numpy(ctx, local_fluxes, gdata, points):
    return q_bulk_at_points(ctx, _ctx_rect(ctx), local_fluxes, points, use_rec=True)


def build_fracture_element_source(ctx, gdata):
    rect = _ctx_rect(ctx)
    local_fluxes = ctx.get("local_fluxes")
    lambda_cell = np.zeros(rect["num_cells"], dtype=float)
    records = []
    if local_fluxes is not None:
        for ci in range(rect["num_cells"]):
            lambda_cell[ci] = float(np.sum(local_fluxes[ci]["source_t_gamma"]))
            if abs(lambda_cell[ci]) > 1.0e-14:
                for s0, s1 in fracture_intervals_in_polygon(rect["cells"][ci]["poly"]):
                    records.append((int(ci), float(s0), float(s1), float(lambda_cell[ci])))
    frac_cell_ids = np.where(np.abs(lambda_cell) > 1.0e-14)[0].astype(np.int32)
    return lambda_cell, frac_cell_ids, records


def build_fracture_cv_source(ctx, gdata):
    rect = _ctx_rect(ctx)
    local_fluxes = ctx.get("local_fluxes")
    lambda_cv = np.zeros(rect["num_vertices"], dtype=float)
    fracture_dof = np.zeros(rect["num_vertices"], dtype=bool)
    if local_fluxes is not None:
        _, lambda_cv, fracture_dof = compute_lce_per_cv_rect(rect, local_fluxes, use_rec=False)
    return lambda_cv, fracture_dof


def compute_lce_per_element_fracture(ctx, local_fluxes, gdata, use_rec=True):
    ctx["local_fluxes"] = local_fluxes
    rect = _ctx_rect(ctx)
    R = compute_lce_per_element_rect(rect, local_fluxes, use_rec=use_rec)
    lambda_cell = np.array([float(np.sum(local_fluxes[ci]["source_t_gamma"])) for ci in range(rect["num_cells"])], dtype=float)
    frac_cell_ids = np.where(np.abs(lambda_cell) > 1.0e-14)[0].astype(np.int32)
    records = []
    for ci in frac_cell_ids:
        for s0, s1 in fracture_intervals_in_polygon(rect["cells"][int(ci)]["poly"]):
            records.append((int(ci), float(s0), float(s1), float(lambda_cell[int(ci)])))
    return R, lambda_cell, frac_cell_ids, records


def compute_lce_per_cv_fracture(ctx, local_fluxes, gdata, use_rec=True):
    ctx["local_fluxes"] = local_fluxes
    return compute_lce_per_cv_rect(_ctx_rect(ctx), local_fluxes, use_rec=use_rec)


def boundary_dof_mask(ctx, tol=1.0e-10):
    return rect_boundary_vertex_mask(_ctx_rect(ctx), tol=tol)


def cv_coordinates(ctx):
    return _ctx_rect(ctx)["vertices_xy"]


def compute_pressure_errors(ctx, local_fluxes, gdata, use_rec=False, degree=None):
    l2, h1, _ = compute_bulk_pressure_flux_errors(ctx, _ctx_rect(ctx), local_fluxes, use_rec=use_rec)
    return l2, h1


def compute_fracture_pressure_errors(ctx, degree=None):
    return _compute_fracture_pressure_errors_rect(ctx)


def compute_p_rec_l2(ctx, local_fluxes, gdata, degree=None):
    return compute_pressure_errors(ctx, local_fluxes, gdata, use_rec=True, degree=degree)[0]


def compute_q_l2_local(ctx, local_fluxes, gdata, use_rec=True, degree=None):
    return compute_bulk_pressure_flux_errors(ctx, _ctx_rect(ctx), local_fluxes, use_rec=use_rec)[2]


def compute_lambda_jump_l2(ctx, local_fluxes, gdata, use_rec=False, tip_frac=TIP_FRAC, eps=None):
    return compute_lambda_jump_l2_rect(ctx, _ctx_rect(ctx), local_fluxes, _ctx_multiplier_data(ctx), use_rec=use_rec, tip_frac=tip_frac, eps=eps)


def improvement_stats(before, after, ids, eps=1.0e-30):
    b = np.abs(np.asarray(before)[ids])
    a = np.abs(np.asarray(after)[ids])
    if len(a) == 0:
        return np.nan, np.nan, np.nan, np.nan
    ratio = a / np.maximum(b, eps)
    return float(np.mean(a < b) * 100.0), float(np.mean(ratio)), float(np.median(ratio)), float(np.max(ratio))


def plot_fracture_lines(ax, gdata):
    ax.plot([gdata["GHOST_START"][0], gdata["GHOST_END"][0]],
            [gdata["GHOST_START"][1], gdata["GHOST_END"][1]],
            "k--", lw=1.0, label=r"$\widetilde{\Gamma}$")
    ax.plot([gdata["FRAC_A"][0], gdata["FRAC_B"][0]],
            [gdata["FRAC_A"][1], gdata["FRAC_B"][1]],
            color="tab:red", lw=2.0, label=r"$\Gamma$")


## 2. Shared hard-curl controls

These controls are identical in all three notebooks.

In [ ]:
import importlib
import sys
_hardcurl_dir = pathlib.Path.cwd() / "fenicsx/code/fracture problem"
if not (_hardcurl_dir / "fracture_hardcurl_common.py").exists():
    _hardcurl_dir = pathlib.Path.cwd()
if str(_hardcurl_dir) not in sys.path:
    sys.path.insert(0, str(_hardcurl_dir))
import fracture_hardcurl_common as hardcurl
importlib.reload(hardcurl)

VARIANT = 'nonconforming_rect'
HARD_REF = REF_DEMO
FACE_GAUSS_ORDER = 32
SOURCE_GAUSS_ORDER = 20
LINE_GAUSS_ORDER = 12
ADAM_STEPS_A = 2000
LBFGS_STEPS_A = 250
ADAM_STEPS_B = 2000
LBFGS_STEPS_B = 150
B_PENALTY_WEIGHTS = (1.0, 10.0, 100.0)

assert hardcurl.DTYPE == np.float64
hardcurl.set_fixed_seeds()
print("float64 convention: PASS")
print("Option B extension-continuity term: N/A (no artificial extension)")

## 3. Gate A0 — isolated line-source jump

This gate is mesh-free and network-free.

In [ ]:
gate_A0 = hardcurl.gate_a0()

## 4. Common dual-CV data and target fluxes

Dual faces crossing the fracture are split once. The split is used for one-sided CG targets and for the discontinuous line-source contribution; the smooth analytic \(q_{p,f}\) itself needs no fracture split.

In [ ]:
problem = hardcurl.build_problem(
    globals(), VARIANT, ref=HARD_REF,
    face_order=FACE_GAUSS_ORDER,
    source_order=SOURCE_GAUSS_ORDER,
    line_order=LINE_GAUSS_ORDER,
)

## 5. Gate A1 — conservation identity before training

The gate tests \(\psi=0\) and a randomly initialized network with matching exact/discrete line-source densities. Every nonempty CV class must satisfy `max|R_xi| <= 1e-13`.

In [ ]:
gate_A1 = hardcurl.gate_a1(problem)
fracture_1d_gate = hardcurl.fracture_1d_conservation_check(problem)

# Option A — line source in the particular field

\(q_A=q_{p,f}+q_{p,\lambda_h}+\nabla^\perp\psi\). The single smooth network has no interface penalty. The analytic trace jump of \(q_{p,\lambda_h}\) is exactly \(\lambda_h\), while the curl term is continuous.

## A2. Train the single stream-function network

In [ ]:
option_A = hardcurl.run_option_a(
    problem, adam_steps=ADAM_STEPS_A, lbfgs_steps=LBFGS_STEPS_A,
    width=48, depth=3, lr=2.0e-3,
)

## A3. Verification battery

Option A is completed and audited before Option B begins. With the \(\lambda_h\)-consistent RHS, conservation should be at machine precision in every class. With exact \(\lambda\), the cut-CV residual is the discrete multiplier consistency error.

In [ ]:
hardcurl.plot_option_a_verification(problem, option_A)
option_A_audit = option_A["audit"]
print("Option A completed: all construction gates passed before Option B.")

# Option B — two subdomain stream functions

\(q_B^\pm=q_{p,f}+\nabla^\perp\psi_\pm\). No \(q_{p,\lambda}\) is used. The multiplier is imposed only through the difference constraint \((q_B^+-q_B^-)\cdot n_\Gamma=\lambda_h\); it is never split 50/50. Each cut data face contributes its one-sided segments. The corner-to-corner geometry has no artificial extension, so that term is reported as N/A.

## B1–B2. Penalty sweep and training

In [ ]:
option_B = hardcurl.run_option_b(
    problem, penalty_weights=B_PENALTY_WEIGHTS,
    adam_steps=ADAM_STEPS_B, lbfgs_steps=LBFGS_STEPS_B,
    width=24, depth=2, lr=2.0e-3,
)

## B3. Verification battery and penalty floor

In [ ]:
hardcurl.plot_verification(problem, option_A, option_B)

## Final A/B comparison

The table reports measured accuracy, conservation, parameter count, and wall time without ranking beyond the numerical evidence.

In [ ]:
comparison_rows = hardcurl.comparison_table(problem, option_A, option_B)

In [ ]:
import json as _json
SUMMARY_DIR = pathlib.Path("result_fracture_MMS_hardcurl_AB")
SUMMARY_DIR.mkdir(exist_ok=True)
mesh_summary = {
    "mesh": VARIANT,
    "A_flux_error": comparison_rows[0]["face_flux_RMSE_exact"],
    "B_flux_error": comparison_rows[1]["face_flux_RMSE_exact"],
    "A_worst_Rxi": comparison_rows[0]["worst_Rxi"],
    "B_worst_Rxi": comparison_rows[1]["worst_Rxi"],
    "A_jump_error": comparison_rows[0]["jump_RMSE_lambda_h"],
    "B_jump_error": comparison_rows[1]["jump_RMSE_lambda_h"],
}
summary_file = SUMMARY_DIR / f"{VARIANT}_summary.json"
summary_file.write_text(_json.dumps(mesh_summary, indent=2) + "\n")
print("Saved", summary_file)

### Conclusion

Option A is conservative by construction with the discrete multiplier-consistent RHS: its line-source particular field carries the exchange distribution and its curl correction cannot change any closed-CV balance. Option B reaches only the penalty-controlled conservation floor shown above; its two smaller networks trade exact jump enforcement for a soft interface constraint. The measured flux errors, parameter counts, and wall times are reported in the table.

## Cross-mesh summary

Run the first two notebooks and save/pass their `comparison_rows` when assembling a publication table. The schema below is fixed across meshes and demonstrates that the reconstruction code and audit columns are unchanged by fracture alignment.

In [ ]:
cross_mesh_rows = []
for _mesh in ("conforming_tri", "nonconforming_tri", "nonconforming_rect"):
    _path = SUMMARY_DIR / f"{_mesh}_summary.json"
    if _path.exists():
        cross_mesh_rows.append(_json.loads(_path.read_text()))
    else:
        print(f"N/A: run the {_mesh} notebook to create {_path.name}")

print("mesh                  A flux RMSE    B flux RMSE    A worst R_xi   B worst R_xi")
for row in cross_mesh_rows:
    print(f"{row['mesh']:<22} {row['A_flux_error']:12.4e} {row['B_flux_error']:12.4e} "
          f"{row['A_worst_Rxi']:13.4e} {row['B_worst_Rxi']:13.4e}")